# Workspace report

What is actually here, in five tables. Useful on a machine you did not set up,
and useful on your own after a month away — the answer to "which of these
seventeen models was the good one" is on disk, not in memory.

In [ ]:
import orinth
import polars as pl

pl.Config.set_tbl_rows(30)
pl.Config.set_fmt_str_lengths(46)

print(orinth.settings.workspace())

## Projects

In [ ]:
pl.DataFrame(
    [
        {"id": project.id, "name": project.name, "tasks": ", ".join(project.task_types)}
        for project in orinth.projects.list()
    ]
)

## Datasets

Sorted by readiness, because a dataset that is not trainable is the one you need
to know about — a `ready` list tells you nothing you were going to act on.

In [ ]:
datasets = pl.DataFrame(
    [
        {
            "id": dataset.id,
            "task": dataset.task_type,
            "items": dataset.total_items,
            "train": dataset.splits.get("train", 0),
            "valid": dataset.splits.get("valid", 0),
            "test": dataset.splits.get("test", 0),
            "state": dataset.readiness.state,
            "trainable": dataset.readiness.trainable,
        }
        for dataset in orinth.datasets.list()
    ]
)
datasets.sort(["trainable", "items"], descending=[False, True])

## Models

In [ ]:
pl.DataFrame(
    [
        {
            "id": model.id,
            "family": model.family,
            "task": model.task_type,
            "source": model.source,
            "available": model.available,
        }
        for model in orinth.models.list()
    ]
).sort(["source", "family"])

## Training runs

`model_id` is the join back to the catalog: a completed run registered that
model, and that is the id every other module wants.

In [ ]:
runs = orinth.train.list()
if runs:
    print(
        pl.DataFrame(
            [
                {
                    "job": run.id[:12],
                    "task": run.task_type,
                    "family": run.model_family,
                    "status": run.status,
                    "percent": run.percent,
                    "model": run.model_id or "",
                }
                for run in runs
            ]
        )
    )
else:
    print("No training jobs yet — the Training page, or orinth.train.start().")

## Evaluations, and what is missing

The interesting cell. Every available model, and whether anyone has scored it —
an unevaluated model is a number nobody has, presented as a fact.

In [ ]:
scored = {job.model_id for job in orinth.evaluate.list() if job.status == "completed"}
rows = [
    {"model": model.id, "task": model.task_type, "evaluated": model.id in scored}
    for model in orinth.models.list()
    if model.available
]
report = pl.DataFrame(rows)
print(report.filter(~pl.col("evaluated")).height, "of", report.height, "models have never been evaluated")
report.sort("evaluated")